# 01 Langgraph Fundamentals

In [2]:
# !pip install langgraph
# !pip install langchain
# !pip install ipywidgets

In [3]:
# ==================================================
# Notebook 01
# LangGraph Fundamentals
# ==================================================

from typing import TypedDict

from langgraph.graph import StateGraph, END

In [4]:
class ResearchState(TypedDict):

    question: str

    notes: list

    answer: str

In [5]:
def researcher_node(state: ResearchState):

    print("Researcher Running...")

    state["notes"].append("Found cloud pricing data")

    return state

In [6]:
def writer_node(state: ResearchState):

    print("Writer Running...")

    state["answer"] = "Cloud pricing report generated."

    return state

In [7]:
graph = StateGraph(ResearchState)

In [8]:
graph.add_node("researcher", researcher_node)

graph.add_node("writer", writer_node)

In [9]:
graph.set_entry_point("researcher")

graph.add_edge("researcher", "writer")

graph.add_edge("writer", END)

In [10]:
app = graph.compile()

In [11]:
initial_state = {"question": "Compare cloud costs", "notes": [], "answer": ""}

In [12]:
result = app.invoke(initial_state)

result

Researcher Running...
Writer Running...


{'question': 'Compare cloud costs',
 'notes': ['Found cloud pricing data'],
 'answer': 'Cloud pricing report generated.'}

In [13]:
class LoopState(TypedDict):

    question: str

    notes: list

    iterations: int

    answer: str

In [14]:
def looping_researcher(state: LoopState):

    state["iterations"] += 1

    state["notes"].append(f"Research Round " f"{state['iterations']}")

    return state

In [15]:
def check_progress(state: LoopState):

    if state["iterations"] >= 3:

        return "writer"

    return "researcher"

In [16]:
def final_writer(state: LoopState):

    state["answer"] = "Research Completed"

    return state

In [17]:
loop_graph = StateGraph(LoopState)

In [18]:
loop_graph = StateGraph(LoopState)

In [19]:
loop_graph.add_node("researcher", looping_researcher)

loop_graph.add_node("writer", final_writer)

In [20]:
loop_graph.set_entry_point("researcher")

In [21]:
loop_graph.add_conditional_edges("researcher", check_progress)

In [22]:
loop_graph.add_edge("writer", END)

In [23]:
loop_app = loop_graph.compile()

In [24]:
result = loop_app.invoke(
    {"question": "Compare cloud costs", "notes": [], "iterations": 0, "answer": ""}
)

result

{'question': 'Compare cloud costs',
 'notes': ['Research Round 1', 'Research Round 2', 'Research Round 3'],
 'iterations': 3,
 'answer': 'Research Completed'}